# H&M M4 — 보호분할(test) 확증

**학습하지 않습니다.** 이미 끝난 체크포인트를 불러 test 분할에서 한 번 채점합니다.

**왜 필요한가.** 지금까지의 모든 수치는 개발분할(validation, 2020-09-02~08)에서 나왔고, 그 구간은 연구 과정에서 반복해서 봤습니다. test 분할(2020-09-09~15)은 **한 번도 계산한 적이 없습니다** — 개발 경로가 `EVAL_TEST=False`를 세 겹으로 고정해뒀습니다.

**한 번뿐입니다.** 열어보고 설정을 바꾸면 test도 개발분할이 됩니다. 그래서 무엇을 보고할지 실행 전에 코드에 고정했습니다 — arm 3개(M1·실제 `q_C` M4·순열 `q_C` M4), 시드 42·43, 주지표 2개(가격·구매금액 가중 적중값@10, 가중 NDCG@10), 판정 조건 3개(M1 대비 우위 / 순열 대비 우위 / 정확도 6개 99% 보호). 결과를 본 뒤 epoch·arm·시드를 고르지 않습니다.

**안전장치.** test를 열기 전에 각 arm을 **validation에서 다시 채점해** 보고된 개발 수치를 오차 `1e-9` 안에서 재현하는지 확인합니다. 재현되지 않으면 보호분할을 열지 않고 멈춥니다 — 체크포인트나 입력이 잘못됐다는 뜻이기 때문입니다. 체크포인트가 없으면 학습으로 넘어가지 않고 중단합니다.

**개발분할 결과 요약(이미 확보):** 두 시드 모두 사전등록 조건 통과. seed 43은 M1 대비 Recall@10 +3.11%, 가중 적중값@10 +6.01%, 가중 NDCG@10 +3.77%이고 순열 대비로도 전부 우위였습니다.

holdout은 열지 않습니다.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
from google.colab import drive

ROOT = Path('/content/drive/MyDrive/논문/data')
DEV = ROOT/'results_v3_hm_m4_k1_assignment_control_hm2y_development_screen_v1'
if not DEV.is_dir():
    if os.path.ismount('/content/drive'):
        raise RuntimeError(f'Drive는 연결됐지만 개발 결과가 없습니다: {DEV}')
    try:
        drive.mount('/content/drive')
    except (ValueError, NotImplementedError) as exc:
        raise RuntimeError('Drive 연결 실패') from exc
    if not DEV.is_dir():
        raise RuntimeError(f'개발 결과가 없습니다: {DEV}')

SOURCE_COMMIT = 'e8198525af1f63aa081d5d5d5cec6fd7c50fc44e'
REPO = Path('/content/clv-m4-confirm-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
if 'lightgcn_clv_v3' in sys.modules:
    assert Path(sys.modules['lightgcn_clv_v3'].__file__).resolve().parent == REPO.resolve(), \
        '다른 소스가 로드되어 있습니다. 런타임을 다시 시작하세요.'
os.chdir(REPO)
sys.path.insert(0, str(REPO))

import lightgcn_clv_m4_hm2y_confirmatory as conf
cfg = conf.configure_confirmatory(dev_out_dir=str(DEV))
print(json.dumps(conf.preflight_summary(cfg), ensure_ascii=False, indent=2))

## 1. 체크포인트가 다 있는지 확인 (아직 열지 않음)

여섯 개(시드 2 × arm 3)가 모두 있어야 합니다. 하나라도 없으면 다음 셀이 중단합니다 — 학습으로 넘어가지 않습니다.

In [ ]:
import lightgcn_clv_m4_k1_assignment_control_hm2y as screen
import lightgcn_clv_m5_economic_positive_weight as legacy

for seed in cfg.seeds:
    folder = conf.discover_arm_hash(cfg.dev_out_dir, seed)   # 없으면 여기서 중단
    root = Path(cfg.dev_out_dir)/'arms'/folder
    print(f'seed {seed} → arms/{folder}')
    for model_id in screen.MODEL_IDS:
        pt = root/f'{model_id}_s{seed}.pt'
        print(f"    {model_id[:54]:<54} {'있음' if pt.is_file() else '없음'}")

## 2. 보호분할 열기 (학습 없음)

`open_test_split=True`를 여기서만 줍니다. 순서는 이렇습니다 — 개발 입력 준비 → 체크포인트 로드 → **validation 재채점으로 개발 결과 재현 확인** → test 분할 구성(고객·상품 수와 학습 행 수를 개발과 대조) → test 채점.

시드당 몇 분 걸립니다. GPU 런타임을 쓰세요.

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU 런타임을 사용하세요.'
frame = conf.run_confirmatory(conf.configure_confirmatory(
    dev_out_dir=str(DEV), open_test_split=True))

## 3. 결과와 원본 ZIP

`per_seed`의 `attribution_pass`가 각 시드의 보호분할 판정입니다. 개발분할에서 두 시드 모두 통과했으니, 여기서도 통과하면 **반복 노출되지 않은 구간에서 확인된 결과**가 됩니다.

**불리하게 나와도 그대로 보고합니다.** 설정을 바꿔 다시 열지 않습니다.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
import pandas as pd

reading = frame.attrs['reading']
print(json.dumps(reading, ensure_ascii=False, indent=2))
print()
show = ['recall@10','ndcg@10','recall@50','ndcg@50'] + list(conf.ECONOMIC_METRICS)
print(frame.set_index(['seed','model_id'])[show].to_string())

paths = frame.attrs['result_paths']
zip_path = Path('/content/m4_hm2y_confirmatory_results.zip')
with ZipFile(zip_path, 'w', compression=ZIP_DEFLATED) as archive:
    for path in paths.values():
        archive.write(path, arcname=Path(path).name)
files.download(str(zip_path))